# Fruit and Vegetable Disease Dataset — Exploratory Analysis

Examines class distribution, image properties, and colour characteristics to inform preprocessing and training decisions.

In [ ]:
# ── Imports & global config ────────────────────────────────────────────────────
import os
import sys
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image, UnidentifiedImageError

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})  # consistent chart rendering defaults

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'OpenCV  : {cv2.__version__}')
print(f'Pillow  : {Image.__version__}')
print(f'Seaborn : {sns.__version__}')

In [ ]:
# ── Dataset root discovery ─────────────────────────────────────────────────────
# Search common relative locations so the notebook works regardless of CWD.
# Drills down single-child directory chains so it correctly resolves:
#   data/Task 2/raw  →  Fruit And Vegetable Diseases Dataset/  →  [class dirs]

SEARCH_PATHS = [
    Path('data/Task 2/raw'),
    Path('../data/Task 2/raw'),
    Path('../../data/Task 2/raw'),
]

# Defined here so find_dataset_root can use it; also referenced in later cells.
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def _dir_has_images(directory: Path) -> bool:
    """True if any direct file child of directory is an image (case-insensitive)."""
    return any(
        f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
        for f in directory.iterdir()
    )

def find_dataset_root(search_paths):
    """Return the directory whose immediate subdirectories contain images.

    Walks down single-child chains (e.g. raw/ → 'Fruit And Vegetable Diseases Dataset/')
    so it doesn't stop one level too early when the zip extracts into a subfolder.
    """
    for base in search_paths:
        if not base.exists():
            continue
        current = base
        for _ in range(6):                       # recurse at most 6 levels
            subdirs = [s for s in current.iterdir() if s.is_dir()]
            if not subdirs:
                break
            # Found class-level root: subdirectories themselves hold image files
            if any(_dir_has_images(s) for s in subdirs):
                return current
            # Single subdir with no images at this level → step into it
            if len(subdirs) == 1 and not _dir_has_images(current):
                current = subdirs[0]
            else:
                break
    return None

DATASET_ROOT = find_dataset_root(SEARCH_PATHS)

if DATASET_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the dataset. '
        'Please set DATASET_ROOT manually to the folder containing the class sub-directories.'
    )

print(f'Dataset root: {DATASET_ROOT.resolve()}')

---
## Section 1 — Folder Structure and Class Discovery

Walks the dataset directory to confirm the layout, naming convention, and whether a train/val/test split already exists.

In [ ]:
# ── Walk directory tree and print structure ────────────────────────────────────
# IMG_EXTENSIONS is defined in the discovery cell above

def print_tree(root: Path, prefix: str = '', max_files: int = 3):
    """Pretty-print a directory tree, summarising large leaf dirs."""
    entries = sorted(root.iterdir())
    dirs  = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file() and e.suffix.lower() in IMG_EXTENSIONS]
    other = [e for e in entries if e.is_file() and e.suffix.lower() not in IMG_EXTENSIONS]

    for i, d in enumerate(dirs):
        connector = '└── ' if (i == len(dirs) - 1 and not files and not other) else '├── '
        n_imgs = sum(1 for f in d.rglob('*') if f.suffix.lower() in IMG_EXTENSIONS)  # rglob counts images in all nested subdirs
        print(f'{prefix}{connector}{d.name}/  [{n_imgs} images]')
        extension = '    ' if connector == '└── ' else '│   '  # match indentation to connector style
        print_tree(d, prefix + extension, max_files)

    if files:
        shown = files[:max_files]
        for f in shown:
            print(f'{prefix}├── {f.name}')
        if len(files) > max_files:
            print(f'{prefix}└── ... and {len(files) - max_files} more image files')
    if other:
        for f in other:
            print(f'{prefix}[non-image] {f.name}')

print(f'{DATASET_ROOT.name}/')
print_tree(DATASET_ROOT)

In [ ]:
# ── Build class registry ───────────────────────────────────────────────────────
# Assumption: each immediate child directory is one class.
# Class label format: "{Produce}__{Quality}"  (double-underscore separator)

class_dirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])

# Gather all image paths per class
class_images = {}
for cd in class_dirs:
    imgs = [f for f in cd.iterdir() if f.suffix.lower() in IMG_EXTENSIONS]
    class_images[cd.name] = imgs

all_classes    = list(class_images.keys())
produce_types  = sorted({c.split('__')[0] for c in all_classes})           # unique produce names (left of double-underscore)
quality_labels = sorted({c.split('__')[1] for c in all_classes if '__' in c})  # ['Healthy', 'Rotten']

print(f'Total classes   : {len(all_classes)}')
print(f'Produce types   : {len(produce_types)}  →  {", ".join(produce_types)}')
print(f'Quality labels  : {quality_labels}')
print()
print('Class list (sorted):')
for c in all_classes:
    print(f'  {c:35s}  {len(class_images[c]):>5} images')

**Findings — Section 1**

The dataset uses a flat single-level layout. Each subdirectory is a class encoded as `{Produce}__{Healthy|Rotten}` (double-underscore separator). There is no pre-existing train/val/test split and no manifest file; all labelling is implicit in the directory name.

---
## Section 2 — Class Distribution

Checks for class imbalance across the 28 classes. A large imbalance ratio typically requires weighted loss or oversampling to prevent the model from ignoring minority classes.

In [ ]:
# ── Per-class counts ─────────────────────────────────────────────────────────────
counts = {c: len(imgs) for c, imgs in class_images.items()}

# Drop any empty class directories before computing statistics
empty_classes = [c for c, n in counts.items() if n == 0]
if empty_classes:
    print(f'⚠  Skipping {len(empty_classes)} empty class director(ies): {empty_classes}')
    counts = {c: n for c, n in counts.items() if n > 0}

sorted_counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True))

total_images    = sum(counts.values())
majority_count  = max(counts.values())
minority_count  = min(counts.values())
imbalance_ratio = majority_count / minority_count if minority_count > 0 else float('inf')  # guard against empty classes

print(f'Total images     : {total_images:,}')
print(f'Majority class   : {max(counts, key=counts.get)} ({majority_count:,} images)')
print(f'Minority class   : {min(counts, key=counts.get)} ({minority_count:,} images)')
print(f'Imbalance ratio  : {imbalance_ratio:.1f}×')

low_count = [c for c, n in counts.items() if n < 100]
if low_count:
    print(f'Classes with < 100 images: {low_count}')
else:
    print('All classes have ≥ 100 images.')

In [ ]:
# ── Bar chart: images per class ────────────────────────────────────────────────
labels   = list(sorted_counts.keys())
values   = list(sorted_counts.values())
colours  = ['#2ecc71' if '__Healthy' in l else '#e74c3c' for l in labels]  # green = healthy, red = rotten

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(labels, values, color=colours, edgecolor='white', linewidth=0.6)

ax.axhline(100, color='black', linestyle='--', linewidth=1.2, label='100-image threshold')
ax.set_title('Image Count per Class', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Images')
ax.set_xlabel('Class')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([l.replace('__', '\n') for l in labels], rotation=45, ha='right', fontsize=8)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            str(val), ha='center', va='bottom', fontsize=7)

healthy_patch = mpatches.Patch(color='#2ecc71', label='Healthy')
rotten_patch  = mpatches.Patch(color='#e74c3c', label='Rotten')
ax.legend(handles=[healthy_patch, rotten_patch,
                   plt.Line2D([0], [0], color='black', linestyle='--', label='100-image threshold')],
          loc='upper right')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()
print('Chart saved as class_distribution.png')

In [ ]:
# ── Fresh vs Rotten split ──────────────────────────────────────────────────────
healthy_total = sum(n for c, n in counts.items() if 'Healthy' in c)
rotten_total  = sum(n for c, n in counts.items() if 'Rotten'  in c)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Pie chart
axes[0].pie(
    [healthy_total, rotten_total],
    labels=[f'Healthy\n({healthy_total:,})', f'Rotten\n({rotten_total:,})'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Overall Healthy vs Rotten Split', fontweight='bold')

# Per-produce grouped bar
produce_healthy = {p: counts.get(f'{p}__Healthy', 0) for p in produce_types}
produce_rotten  = {p: counts.get(f'{p}__Rotten',  0) for p in produce_types}

x = np.arange(len(produce_types))
w = 0.38  # bar width; narrow enough to leave a visible gap between healthy/rotten pairs
axes[1].bar(x - w/2, [produce_healthy[p] for p in produce_types], w,
            label='Healthy', color='#2ecc71', edgecolor='white')
axes[1].bar(x + w/2, [produce_rotten[p]  for p in produce_types], w,
            label='Rotten',  color='#e74c3c', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(produce_types, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Images')
axes[1].set_title('Healthy vs Rotten per Produce Type', fontweight='bold')
axes[1].legend()

plt.suptitle('Fresh / Rotten Split Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fresh_rotten_split.png', bbox_inches='tight')
plt.show()

print(f'Healthy : {healthy_total:,}  ({100*healthy_total/total_images:.1f}%)')
print(f'Rotten  : {rotten_total:,}  ({100*rotten_total/total_images:.1f}%)')
h_r_ratio = max(healthy_total, rotten_total) / min(healthy_total, rotten_total)
print(f'Healthy/Rotten imbalance ratio: {h_r_ratio:.2f}×')

---
## Section 3 — Image Property Audit

Samples 200 images to check resolution distribution and colour mode. Large resolution variance affects the choice of resize strategy; non-RGB images require conversion before being passed to the model.

In [ ]:
# ── Build a flat list of all image paths ───────────────────────────────────────
all_paths = [(cls, p) for cls, paths in class_images.items() for p in paths]
print(f'Total image paths found: {len(all_paths):,}')

# Sample 200 for the property audit
AUDIT_SAMPLE = 200
sample_paths = random.sample(all_paths, min(AUDIT_SAMPLE, len(all_paths)))  # min() handles datasets smaller than AUDIT_SAMPLE
print(f'Audit sample size: {len(sample_paths)}')

In [ ]:
# ── Audit each sampled image ───────────────────────────────────────────────────
widths, heights = [], []
mode_counts = defaultdict(int)
corrupt_files = []

for cls, path in sample_paths:
    try:
        with Image.open(path) as img:
            img.verify()          # detect truncated / corrupt files
        with Image.open(path) as img:  # must re-open after verify(); verify() exhausts the file handle
            w, h = img.size
            mode_counts[img.mode] += 1
            widths.append(w)
            heights.append(h)
    except (UnidentifiedImageError, Exception) as e:
        corrupt_files.append((str(path), str(e)))

print('=== Image Size Statistics (sample of 200) ===')
print(f'Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.1f}  std: {np.std(widths):.1f}')
print(f'Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.1f}  std: {np.std(heights):.1f}')
print()
print('=== Colour Mode Breakdown ===')
for mode, cnt in sorted(mode_counts.items(), key=lambda x: -x[1]):
    flag = '' if mode == 'RGB' else '  ⚠  NON-RGB'
    print(f'  {mode:10s}: {cnt:>4}{flag}')
print()
if corrupt_files:
    print(f'⚠  Corrupt / unreadable files ({len(corrupt_files)}):')
    for p, err in corrupt_files:
        print(f'   {p}  →  {err}')
else:
    print('✓  No corrupt files detected in the sample.')

In [ ]:
# ── Histogram of image dimensions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(widths,  bins=30, color='#3498db', edgecolor='white', linewidth=0.5)
axes[0].set_title('Width Distribution')
axes[0].set_xlabel('Pixels')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean={np.mean(widths):.0f}px')
axes[0].legend(fontsize=8)

axes[1].hist(heights, bins=30, color='#9b59b6', edgecolor='white', linewidth=0.5)
axes[1].set_title('Height Distribution')
axes[1].set_xlabel('Pixels')
axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean={np.mean(heights):.0f}px')
axes[1].legend(fontsize=8)

axes[2].scatter(widths, heights, alpha=0.35, s=15, color='#e67e22')
axes[2].set_title('Width vs Height Scatter')
axes[2].set_xlabel('Width (px)')
axes[2].set_ylabel('Height (px)')
axes[2].axline((0, 0), slope=1, color='gray', linestyle=':', linewidth=1, label='Square')  # diagonal reference; square images lie on this line
axes[2].legend(fontsize=8)

plt.suptitle('Image Dimension Audit (200-image sample)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('image_dimensions.png', bbox_inches='tight')
plt.show()

---
## Section 4 — Sample Image Visualisation

Displays three examples per class to catch systematic issues (watermarks, mislabels, inconsistent backgrounds) that automated statistics cannot detect.

In [ ]:
# ── Display 3 sample images per class ─────────────────────────────────────────
N_SAMPLES_PER_CLASS = 3
n_classes = len(all_classes)
n_cols    = N_SAMPLES_PER_CLASS
n_rows    = n_classes

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.8, n_rows * 2.6))

for row_idx, cls in enumerate(all_classes):
    imgs_for_cls = class_images[cls]
    sampled = random.sample(imgs_for_cls, min(N_SAMPLES_PER_CLASS, len(imgs_for_cls)))

    quality   = 'Healthy' if 'Healthy' in cls else 'Rotten'
    produce   = cls.split('__')[0]
    col_color = '#1a7a4a' if quality == 'Healthy' else '#9b1c1c'  # dark green / dark red for the row label

    for col_idx in range(N_SAMPLES_PER_CLASS):
        ax = axes[row_idx][col_idx]
        if col_idx < len(sampled):
            try:
                img = Image.open(sampled[col_idx]).convert('RGB')
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, 'Error', ha='center', va='center', transform=ax.transAxes)
        else:
            ax.axis('off')
            continue

        ax.axis('off')
        if col_idx == 0:
            ax.set_title(f'{produce}\n{quality}', fontsize=8, fontweight='bold',
                         color=col_color, loc='left', pad=3)

plt.suptitle('Sample Images — 3 per Class (28 classes × 2 quality states)',
             fontsize=13, fontweight='bold', y=1.002)
plt.tight_layout(h_pad=0.5, w_pad=0.3)
plt.savefig('sample_images.png', bbox_inches='tight', dpi=100)
plt.show()
print('Grid saved as sample_images.png')

---
## Section 5 — Colour Distribution (HSV)

Computes per-class average HSV histograms to assess whether the Hue, Saturation, or Value channels separate healthy from rotten produce. A high intersection distance indicates a useful colour signal.

In [ ]:
# ── Compute mean HSV histograms per class ──────────────────────────────────────
HSV_SAMPLE = 100   # images per class (all if fewer available)
HIST_BINS  = 64

hsv_hists = {}     # class → {'H': array, 'S': array, 'V': array}

for cls in all_classes:
    paths = class_images[cls]
    sample = random.sample(paths, min(HSV_SAMPLE, len(paths)))

    H_acc = np.zeros(HIST_BINS)
    S_acc = np.zeros(HIST_BINS)
    V_acc = np.zeros(HIST_BINS)
    n_ok  = 0

    for p in sample:
        try:
            bgr = cv2.imread(str(p))
            if bgr is None:
                continue  # cv2.imread returns None for unreadable files rather than raising
            hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
            H_acc += cv2.calcHist([hsv], [0], None, [HIST_BINS], [0, 180]).flatten()
            S_acc += cv2.calcHist([hsv], [1], None, [HIST_BINS], [0, 256]).flatten()
            V_acc += cv2.calcHist([hsv], [2], None, [HIST_BINS], [0, 256]).flatten()
            n_ok  += 1
        except Exception:
            continue

    if n_ok > 0:
        hsv_hists[cls] = {
            'H': H_acc / (n_ok * (180 / HIST_BINS)),   # normalise to density
            'S': S_acc / (n_ok * (256 / HIST_BINS)),
            'V': V_acc / (n_ok * (256 / HIST_BINS)),
            'n': n_ok
        }

print(f'HSV histograms computed for {len(hsv_hists)} classes (sample={HSV_SAMPLE} per class).')

In [ ]:
# ── Plot 1: Aggregate Healthy vs Rotten HSV distributions ─────────────────────
# Average across all produce types to see the overall fresh/rotten colour signal.

def aggregate_hists(label_keyword):
    hists = [v for k, v in hsv_hists.items() if label_keyword in k]
    if not hists:
        return None, None, None
    H = np.mean([h['H'] for h in hists], axis=0)
    S = np.mean([h['S'] for h in hists], axis=0)
    V = np.mean([h['V'] for h in hists], axis=0)
    return H, S, V

h_H, h_S, h_V = aggregate_hists('Healthy')
r_H, r_S, r_V = aggregate_hists('Rotten')

bins_H  = np.linspace(0, 180, HIST_BINS)  # Hue spans 0–180 in OpenCV's HSV convention
bins_SV = np.linspace(0, 256, HIST_BINS)  # Saturation and Value span 0–255

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, h_data, r_data, bins, channel, xlabel in [
    (axes[0], h_H, r_H, bins_H,  'Hue',        'Hue (0–180°)'),
    (axes[1], h_S, r_S, bins_SV, 'Saturation', 'Saturation (0–255)'),
    (axes[2], h_V, r_V, bins_SV, 'Value',      'Value / Brightness (0–255)'),
]:
    ax.plot(bins, h_data, color='#2ecc71', linewidth=2, label='Healthy')
    ax.fill_between(bins, h_data, alpha=0.25, color='#2ecc71')
    ax.plot(bins, r_data, color='#e74c3c', linewidth=2, label='Rotten', linestyle='--')
    ax.fill_between(bins, r_data, alpha=0.25, color='#e74c3c')
    ax.set_title(f'{channel} Channel', fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Mean Pixel Density')
    ax.legend(fontsize=9)

plt.suptitle('Average HSV Histogram: Healthy vs Rotten (all produce types aggregated)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('hsv_healthy_vs_rotten.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Per-produce-type H channel overlay ─────────────────────────────────
# Shows whether colour separation is consistent across produce types or
# produce-specific (e.g., Banana yellowing vs Apple browning)

n_produce = len(produce_types)
cols = 4
rows = (n_produce + cols - 1) // cols  # ceiling division to fit all produce types

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3), sharey=False)
axes_flat = axes.flatten()

for idx, produce in enumerate(produce_types):
    ax = axes_flat[idx]
    h_cls = f'{produce}__Healthy'
    r_cls = f'{produce}__Rotten'

    if h_cls in hsv_hists:
        ax.plot(bins_H, hsv_hists[h_cls]['H'], color='#2ecc71', linewidth=2, label='Healthy')
        ax.fill_between(bins_H, hsv_hists[h_cls]['H'], alpha=0.2, color='#2ecc71')
    if r_cls in hsv_hists:
        ax.plot(bins_H, hsv_hists[r_cls]['H'], color='#e74c3c', linewidth=2,
                linestyle='--', label='Rotten')
        ax.fill_between(bins_H, hsv_hists[r_cls]['H'], alpha=0.2, color='#e74c3c')

    ax.set_title(produce, fontweight='bold', fontsize=9)
    ax.set_xlabel('Hue (0–180°)', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

for idx in range(n_produce, len(axes_flat)):
    axes_flat[idx].set_visible(False)  # hide unused grid cells

plt.suptitle('Hue Distribution per Produce Type — Healthy vs Rotten',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hsv_per_produce.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Saturation & Value overlay per produce type ────────────────────────────────
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3), sharey=False)
axes_flat = axes.flatten()

for idx, produce in enumerate(produce_types):
    ax = axes_flat[idx]
    h_cls = f'{produce}__Healthy'
    r_cls = f'{produce}__Rotten'

    if h_cls in hsv_hists:
        ax.plot(bins_SV, hsv_hists[h_cls]['S'], color='#2ecc71', linewidth=1.5, label='H-Sat')
        ax.plot(bins_SV, hsv_hists[h_cls]['V'], color='#27ae60', linewidth=1.5,
                linestyle=':', label='H-Val')
    if r_cls in hsv_hists:
        ax.plot(bins_SV, hsv_hists[r_cls]['S'], color='#e74c3c', linewidth=1.5,
                linestyle='--', label='R-Sat')
        ax.plot(bins_SV, hsv_hists[r_cls]['V'], color='#c0392b', linewidth=1.5,
                linestyle='-.', label='R-Val')

    ax.set_title(produce, fontweight='bold', fontsize=9)
    ax.set_xlabel('S / V (0–255)', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.legend(fontsize=6, ncol=2)
    ax.tick_params(labelsize=7)

for idx in range(n_produce, len(axes_flat)):
    axes_flat[idx].set_visible(False)  # hide unused grid cells

plt.suptitle('Saturation & Value Distributions per Produce Type — Healthy vs Rotten',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sv_per_produce.png', bbox_inches='tight')
plt.show()